In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================
print("Loading imports...")
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from skopt import BayesSearchCV
from skopt.space import Real

# =========================================================
# 2. LOAD DATA
# =========================================================
print("Loading data...")
# parse_dates ensures date columns are loaded as datetime objects, saving a conversion step later
train = pd.read_csv('store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
stores = pd.read_csv('store-sales-time-series-forecasting/stores.csv')
holidays_events = pd.read_csv('store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])

# =========================================================
# 3. DATA MERGING & PREPARATION
# =========================================================
print("Merging data...")
# Merge store metadata (city, state, type, cluster) into our main sets
train = train.merge(stores, on='store_nbr', how='left')

# Prepare Holidays
# In Ecuador, if a holiday falls on a weekend, it is often 'transferred' to a weekday.
# We filter out the original date of transferred holidays so we don't count them twice.
valid_holidays = holidays_events[holidays_events['transferred'] == False].drop_duplicates(subset=['date'])
valid_holidays = valid_holidays[['date', 'type']].rename(columns={'type': 'holiday_type'})

train = train.merge(valid_holidays, on='date', how='left')

# Copy 'train' to 'df' to maintain naming consistency with other notebooks.
# No test data is concatenated in this specific workflow.
df = train.copy()

# The competition metric is Root Mean Squared Logarithmic Error (RMSLE).
# By applying log1p (log(1 + x)) to the target now, we can just use standard 
# RMSE as our loss function in the model.
df['log1p_sales'] = np.log1p(df['sales'])

# =========================================================
# 4. FEATURE ENGINEERING
# =========================================================
print("Engineering features...")

# --- Date & Fourier Features ---
# Day of week is strictly categorical
df['dayofweek'] = df['date'].dt.dayofweek.astype(str)
df['dayofyear'] = df['date'].dt.dayofyear

# Fourier terms (sin/cos) model annual seasonality effectively. 
for i in range(1, 4):
    df[f'sin_{i}'] = np.sin(2 * np.pi * i * df['dayofyear'] / 365.25)
    df[f'cos_{i}'] = np.cos(2 * np.pi * i * df['dayofyear'] / 365.25)

# --- Earthquake Feature ---
# The devastating earthquake occurred on April 16, 2016. It drastically affected 
# supermarket sales (panic buying, relief supplies) for ~4 weeks.
df['earthquake_impact'] = ((df['date'] >= '2016-04-16') & (df['date'] <= '2016-05-16')).astype(int)

# --- Payday Feature ---
# Wages in the public sector are paid every 15th and on the last day of the month.
# Sales spike during these times. This calculates days since the most recent payday.
df['day'] = df['date'].dt.day
df['is_month_end'] = df['date'].dt.is_month_end
df['days_since_payday'] = np.where(
    df['day'] < 15, 
    df['day'],  # Logic for the 1st through the 14th
    np.where(
        df['day'] == 15,
        0,          # The 15th is payday
        np.where(
            df['is_month_end'],
            0,          # The last day of the month is payday
            df['day'] - 15  # Logic for the 16th up to the day before month-end
        )
    )
)
df = df.drop(columns=['day', 'is_month_end'])

# --- Holiday Features ---
# Convert holiday presence into a binary feature
df['is_holiday'] = df['holiday_type'].notnull().astype(int)

# --- Interaction Terms ---
# We do not use interaction terms since we
# are mostly interested in which lags are the most significant
# df['family_promo_interaction'] = df['family'].astype(str) + "_promo_" + df['onpromotion'].astype(str)

# --- Global Lag Features (16-31 Days) ---
# Group by store and family, then shift the target variable backwards.
# The Kaggle test set is 15 days long, so we use 16+ day lags to avoid 
# the need for recursive predictions.
lag_days = list(range(16,32))
for lag in lag_days:
    df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['log1p_sales'].shift(lag)

# Shifting creates NaNs at the start of the timeline. We fill with 0 temporarily.
# We will drop the first 31 days of the training set later to prevent training on this artificial data.
lag_cols = [f'lag_{lag}' for lag in lag_days]
df[lag_cols] = df[lag_cols].fillna(0)


# =========================================================
# 5. SETUP THE PIPELINE
# =========================================================
# Explicitly define feature types so the ColumnTransformer knows how to handle them
categorical_cols = ['store_nbr', 'family', 'city', 'state', 'dayofweek'] #, 'family_promo_interaction']
numeric_cols = [
    'onpromotion', 'sin_1', 'cos_1', 'sin_2', 'cos_2', 'sin_3', 'cos_3', 
    'earthquake_impact', 'days_since_payday', 'is_holiday'
] + lag_cols

preprocessor = ColumnTransformer(
    transformers=[
        # sparse_output=True is memory efficient. handle_unknown='ignore' prevents 
        # crashes if a new category unexpectedly appears during inference.
        # We do not use the drop parameter of OneHotEncoder since we will use Lasso which will handle the multi-collinearity.
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lasso', Lasso(max_iter=10000)) #make sure it converges
])

# =========================================================
# 6. CREATE X AND Y DATAFRAMES
# =========================================================
print("Splitting data...")
# Copy 'df' to 'train_full' to maintain naming consistency with other notebooks.
train_full = df.copy()

# Filter out the first 31 days of training data since the lag features are 0 (unreliable)
train_full = train_full[train_full['date'] >= train_full['date'].min() + pd.Timedelta(days=31)]

X_full_train = train_full.drop(columns=['log1p_sales', 'sales'])
y_full_train = train_full['log1p_sales']

# =========================================================
# 7. HYPERPARAMETER TUNING (BAYES SEARCH)
# =========================================================
print("Setting up BayesSearchCV...")
search_space = {
    'lasso__alpha': Real(1e-2, 1e4, prior='log-uniform')
}

tscv = TimeSeriesSplit(n_splits=5)

bayes_search = BayesSearchCV(
    estimator=model_pipeline,
    search_spaces=search_space,
    n_iter=15,                              
    cv=tscv,                                
    scoring='neg_root_mean_squared_error',  
    n_jobs=-1,                              
    random_state=0,
    verbose=0
)

print("Running Bayesian Optimization to find best alpha...")
bayes_search.fit(X_full_train, y_full_train)

best_alpha = bayes_search.best_params_['lasso__alpha']
print(f"\nBest Alpha found: {best_alpha:.4f}")
print(f"Best CV RMSE: {-bayes_search.best_score_:.4f}")

# Extract the tuned model
best_model_pipeline = bayes_search.best_estimator_

# =========================================================
# 8. COEFFICIENT ANALYSIS (FINDING BEST LAGS)
# =========================================================
print("\nExtracting feature coefficients...")

# Access the fitted preprocessor and the fitted Lasso model from the pipeline
fitted_preprocessor = best_model_pipeline.named_steps['preprocessor']
fitted_lasso = best_model_pipeline.named_steps['lasso']

# Extract the dynamically generated feature names
feature_names = fitted_preprocessor.get_feature_names_out()

# Extract the coefficients
coefficients = fitted_lasso.coef_

# Combine into a readable Pandas DataFrame
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Add an absolute value column for sorting importance
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()

# Filter for just the 'lag' features
lag_coefs = coef_df[coef_df['Feature'].str.contains('lag_')].copy()

print("\n--- Unsorted Lag Features ---")
print(lag_coefs)

# Sort the 'lag' features by their absolute values
best_lags = lag_coefs.sort_values(by='Abs_Coefficient', ascending=False)

print("\n--- Most Important (Sorted) Lag Features ---")
print(best_lags)

# View coefficients that Lasso zeroed out (dropped)
zeroed_lags = lag_coefs[lag_coefs['Coefficient'] == 0]
print(f"\nLasso dropped {len(zeroed_lags)} lag features entirely.")